# 05 – Where do the image features land in 3D?

**Question:** a 3D VLM (Qwen-3D, LLaVA-3D) takes one feature $f_i$ per visual token from the 2D vision encoder,
back-projects it with depth to a 3D point $p_i$, and merges features per voxel (average pooling). Only voxels that
received at least one feature become tokens – in their wording these are the "occupied" voxels.

Here we only look at **positions**, so no vision encoder is needed – only the token grid, depth, intrinsics and poses.
We compare two ways to place a token in 3D:

| variant | 3D points per token | meaning |
|---|---|---|
| **center** | 1 – depth at the token center | our reading of Qwen-3D ("unproject their corresponding depth values") |
| **9 samples** | 3 × 3 – depth at a 3 × 3 grid inside the token | our reading of Peter's aggregation (**his code is not pushed yet – replace once it is**) |

and check where these points lie relative to **our own TSDF labels** (occupied / free / behind / unknown, notebook 04).

**Token grid (Qwen2.5-VL):** 14 px ViT patches, merged 2 × 2 → one token per 28 × 28 px of the *resized* image.
The processor resizes so that height and width are multiples of 28 and the pixel count lies in
[`MIN_PIXELS`, `MAX_PIXELS`] (`smart_resize` below, re-implemented from `qwen_vl_utils` – verify against the
settings Qwen-3D actually uses). For 1920 × 1440 iPhone frames and the default `MAX_PIXELS` this gives a
41 × 30 grid = 1,230 tokens per frame. Each token covers ≈ 6 × 6 pixels of the 256 × 192 depth map.

Sections: 1 setup + TSDF · 2 token positions · 3 mesh view · 4 voxel view · 5 center vs. 9 samples.

In [ ]:
%matplotlib inline
import math
import time
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
import open3d as o3d
import plotly.graph_objects as go
from scipy import ndimage

from volumetric_reasoning.config import load_config, resolve_path
from volumetric_reasoning.datasets import ScanNetPPIphoneScene
from volumetric_reasoning.geometry.tsdf import (DenseTSDF, bounds_from_points, pool_states,
                                                UNKNOWN, FREE, OCCUPIED, BEHIND, STATE_NAMES)
from volumetric_reasoning.geometry.backprojection import frames_to_points, backproject_pixels, transform_points

# ---- settings -----------------------------------------------------------------
SCENE_ID = None             # None = dataset.scene_id from the config (1ada7a0617, the clean office)
N_TSDF_FRAMES = None        # None = every tsdf.frame_stride-th frame, as in notebook 04
N_FEATURE_FRAMES = 16       # frames the VLM sees, spread uniformly over the video
VOXEL_SIZE = 0.02           # TSDF resolution (m) for the labels
W_MIN = 3                   # min. observations for a TSDF voxel to count as known
PATCH = 28                  # Qwen2.5-VL: 14 px patches merged 2x2 -> one token per 28x28 px
MIN_PIXELS, MAX_PIXELS = 56 * 56, 28 * 28 * 1280   # Qwen2.5-VL processor defaults
SAMPLES_PER_SIDE = 3        # 3 -> 3x3 = 9 samples per token
TOKEN_VOXEL = 0.08          # voxel size for merging features into tokens (multiple of VOXEL_SIZE)
MAX_HEIGHT = 2.0            # crop the ceiling in the 3D views (None = everything)
SAVE_HTML = False           # also write the plotly views as standalone HTML to outputs/<scene>/
# ----------------------------------------------------------------------------------

cfg = load_config()
if SCENE_ID is not None:
    cfg["dataset"]["scene_id"] = SCENE_ID
tc, dc = cfg["tsdf"], cfg["depth"]
scene = ScanNetPPIphoneScene(resolve_path(cfg["dataset"]["root"]), cfg["dataset"]["scene_id"])
out_dir = resolve_path(cfg["output"]["root"]) / scene.scene_id
assert np.isclose(TOKEN_VOXEL / VOXEL_SIZE, round(TOKEN_VOXEL / VOXEL_SIZE)), "TOKEN_VOXEL must be a multiple of VOXEL_SIZE"
FACTOR = int(round(TOKEN_VOXEL / VOXEL_SIZE))

# Same colors as notebook 04, plus "outside" for points outside the TSDF grid.
OUTSIDE = 4
LABEL_NAMES = {**STATE_NAMES, OUTSIDE: "outside grid"}
LABEL_HEX = {UNKNOWN: "#d9d9d9", FREE: "#9ecae1", OCCUPIED: "#d62728", BEHIND: "#fdae6b", OUTSIDE: "#555555"}

## 1. Our TSDF reconstruction (as in notebook 04)

In [ ]:
tsdf_idx = scene.select_uniform(N_TSDF_FRAMES) if N_TSDF_FRAMES else scene.select_frames(stride=tc["frame_stride"])
tsdf_frames = scene.load_frames(tsdf_idx, with_rgb=False)
pts, _, _ = frames_to_points(tsdf_frames[:: max(1, len(tsdf_frames) // 70)], dc["min_depth"], dc["max_depth"])
lo, hi = bounds_from_points(pts, margin=0.1)

grid = DenseTSDF.from_bounds(lo, hi, VOXEL_SIZE, tc["trunc_voxels"])
t0 = time.time()
grid.integrate_frames(tsdf_frames, dc["min_depth"], dc["max_depth"])
state = grid.classify(W_MIN)
mesh = grid.extract_mesh(W_MIN)
print(f"{scene.scene_id}: TSDF from {len(tsdf_frames)} frames in {time.time() - t0:.0f} s, grid {grid.dims}, "
      f"mesh {len(mesh.triangles):,} triangles")

## 2. Token positions: center vs. 9 samples

For every feature frame and every token we compute sample positions in depth-pixel coordinates, read the depth at
the pixel containing each sample, and back-project. Samples with invalid depth (0, or outside
[`min_depth`, `max_depth`]) are dropped; a token with no valid sample gets no position at all.

In [ ]:
def smart_resize(h, w, factor=PATCH, min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS):
    # Image size the Qwen2.5-VL processor resizes to (re-implementation of qwen_vl_utils.smart_resize).
    h_bar = max(factor, round(h / factor) * factor)
    w_bar = max(factor, round(w / factor) * factor)
    if h_bar * w_bar > max_pixels:
        beta = math.sqrt(h * w / max_pixels)
        h_bar = max(factor, math.floor(h / beta / factor) * factor)
        w_bar = max(factor, math.floor(w / beta / factor) * factor)
    elif h_bar * w_bar < min_pixels:
        beta = math.sqrt(min_pixels / (h * w))
        h_bar = math.ceil(h * beta / factor) * factor
        w_bar = math.ceil(w * beta / factor) * factor
    return h_bar, w_bar


def token_samples(rgb_hw, depth_hw, n):
    # (T, n*n, 2) sample positions (u, v) in depth-pixel coordinates; n = 1 is the token center.
    # The resize keeps the full field of view, so the token grid spans the whole depth map.
    gh, gw = (s // PATCH for s in smart_resize(*rgb_hw))
    offs = (np.arange(n) + 0.5) / n                                   # sub-cell centers, in token units
    r, c = np.meshgrid(np.arange(gh), np.arange(gw), indexing="ij")
    oy, ox = np.meshgrid(offs, offs, indexing="ij")
    v = (r.reshape(-1, 1) + oy.reshape(1, -1)) * depth_hw[0] / gh
    u = (c.reshape(-1, 1) + ox.reshape(1, -1)) * depth_hw[1] / gw
    return np.stack([u, v], axis=-1), (gh, gw)


def token_points(frames, n):
    # World points (T_total, n*n, 3) with NaN for invalid depth, and the frame number of each token.
    rgb_hw = (scene.rgb_height, scene.rgb_width)
    all_pts, all_frame = [], []
    for k, f in enumerate(frames):
        uv, _ = token_samples(rgb_hw, f.depth.shape, n)
        u, v = uv[..., 0].ravel(), uv[..., 1].ravel()
        ui = np.clip(np.floor(u).astype(int), 0, f.depth.shape[1] - 1)
        vi = np.clip(np.floor(v).astype(int), 0, f.depth.shape[0] - 1)
        d = f.depth[vi, ui].astype(np.float64)
        ok = np.isfinite(d) & (d > dc["min_depth"]) & (d <= dc["max_depth"])
        p = np.full((len(d), 3), np.nan)
        p[ok] = transform_points(backproject_pixels(u[ok], v[ok], d[ok], f.K_depth), f.cam_to_world)
        all_pts.append(p.reshape(len(uv), n * n, 3))
        all_frame.append(np.full(len(uv), k))
    return np.concatenate(all_pts), np.concatenate(all_frame)


def label_of(points):
    # TSDF label of the 2 cm voxel containing each point (OUTSIDE if outside the grid).
    idx = np.floor((points - grid.origin) / grid.voxel_size).astype(int)
    inside = ((idx >= 0) & (idx < np.array(grid.dims))).all(1)
    lab = np.full(len(points), OUTSIDE, np.uint8)
    lab[inside] = state[tuple(idx[inside].T)]
    return lab


def spread(p):
    # Largest distance between two valid samples of the same token (0 for a single sample).
    d = np.linalg.norm(p[:, :, None] - p[:, None, :], axis=-1)
    return np.nanmax(np.where(np.isnan(d), -np.inf, d), axis=(1, 2)).clip(min=0)


feat_frames = scene.load_frames(scene.select_uniform(N_FEATURE_FRAMES), with_rgb=False)
variants = {}
for name, n in [("center", 1), ("9 samples", SAMPLES_PER_SIDE)]:
    P, frame_of = token_points(feat_frames, n)
    valid = ~np.isnan(P[..., 0])                  # (T, n*n)
    has_pos = valid.any(1)
    pts = P[valid]                                # (N, 3) all valid samples
    tok = np.nonzero(valid)[0]                    # token id of each sample
    variants[name] = dict(P=P, valid=valid, pts=pts, tok=tok, frame=frame_of[tok], label=label_of(pts),
                          spread=spread(P[has_pos]) if n > 1 else None, has_pos=has_pos)
    print(f"{name:9s}: {len(P):,} tokens ({len(P) // len(feat_frames)} per frame), "
          f"{has_pos.mean() * 100:.1f} % with a position, {len(pts):,} 3D points")

### Where do the points land? (label of the 2 cm TSDF voxel containing each point)

Points come from **single-frame** depth, labels from the **fused** TSDF, so many points land next to the surface
voxel instead of in it. "occupied ± k voxels" therefore also counts points within k voxels of an occupied voxel.

**Expect a lot of "free" even without any error:** on 1ada7a0617, even the raw depth pixels of the frames that *built*
the TSDF land ~33 % in free and ~17 % in behind-surface voxels. The fused distance $D\mu$ at a surface point's voxel
scatters by ±2–3 cm (noise, averaging over views, projective distance inflated on slanted surfaces like the floor),
while the occupied band is only ±1.73 cm (half voxel diagonal). So use the "± 1/2 voxels" columns to judge whether
features sit on our surface; points far from any occupied voxel are the real outliers (flying pixels, pose errors).

In [ ]:
occ_near = {k: ndimage.binary_dilation(state == OCCUPIED, iterations=k) for k in (1, 2)}

def near_occupied(points, k):
    idx = np.floor((points - grid.origin) / grid.voxel_size).astype(int)
    inside = ((idx >= 0) & (idx < np.array(grid.dims))).all(1)
    out = np.zeros(len(points), bool)
    out[inside] = occ_near[k][tuple(idx[inside].T)]
    return out

print(f"{'':10s}" + "".join(f"{LABEL_NAMES[s]:>16s}" for s in LABEL_NAMES) + f"{'occupied ±1 vox':>17s}{'occupied ±2 vox':>17s}")
for name, v in variants.items():
    v["near_occ"] = near_occupied(v["pts"], 1)
    v["near_occ2"] = near_occupied(v["pts"], 2)
    row = "".join(f"{(v['label'] == s).mean() * 100:15.1f}%" for s in LABEL_NAMES)
    print(f"{name:10s}{row}{v['near_occ'].mean() * 100:16.1f}%{v['near_occ2'].mean() * 100:16.1f}%")

## 3. Mesh view: points on our reconstructed mesh

Gray: our TSDF mesh (decimated for the browser). Points: back-projected token positions. `COLOR_BY`:

* `"label"` – TSDF label at the point (red occupied, blue free, orange behind, gray unknown, dark outside)
* `"frame"` – which feature frame the token came from
* `"spread"` – only for 9 samples: spread of the point's token (yellow = samples far apart = token crosses a depth edge)

Plotly view: drag to rotate, scroll to zoom, double-click to reset; click legend entries to hide them.
`SHOW_O3D = True` opens the same scene in an Open3D window instead.

In [ ]:
VARIANT = "center"      # "center" or "9 samples"
COLOR_BY = "label"         # "label", "frame" or "spread"
MAX_POINTS = 60_000        # random subset for the browser
MESH_TRIANGLES = 150_000
SHOW_O3D = True

def crop_mask(p):
    return np.ones(len(p), bool) if MAX_HEIGHT is None else p[:, 2] < MAX_HEIGHT

def mesh_trace(m, n_tri=MESH_TRIANGLES):
    m = o3d.geometry.TriangleMesh(m)
    if MAX_HEIGHT is not None:
        lo_b, hi_b = m.get_min_bound(), m.get_max_bound()
        m = m.crop(o3d.geometry.AxisAlignedBoundingBox(lo_b, [hi_b[0], hi_b[1], MAX_HEIGHT]))
    if len(m.triangles) > n_tri:
        m = m.simplify_quadric_decimation(n_tri)
    V, F = np.asarray(m.vertices), np.asarray(m.triangles)
    return go.Mesh3d(x=V[:, 0], y=V[:, 1], z=V[:, 2], i=F[:, 0], j=F[:, 1], k=F[:, 2], color="#bdbdbd",
                     opacity=1.0, name="TSDF mesh", showlegend=True, hoverinfo="skip",
                     lighting=dict(ambient=0.55, diffuse=0.6, specular=0.05), lightposition=dict(x=1e3, y=-1e3, z=3e3))

def point_traces(v, color_by):
    keep = np.nonzero(crop_mask(v["pts"]))[0]
    if len(keep) > MAX_POINTS:
        keep = np.sort(np.random.default_rng(0).choice(keep, MAX_POINTS, replace=False))
    p = v["pts"][keep]
    style = dict(mode="markers", marker=dict(size=2))
    if color_by == "label":
        lab = v["label"][keep]
        return [go.Scatter3d(x=p[lab == s, 0], y=p[lab == s, 1], z=p[lab == s, 2], name=f"in {LABEL_NAMES[s]} ({(lab == s).sum():,})",
                             mode="markers", marker=dict(size=2, color=LABEL_HEX[s])) for s in LABEL_NAMES if (lab == s).any()]
    if color_by == "frame":
        c, title = v["frame"][keep], "feature frame"
    elif color_by == "spread":
        if v["spread"] is None:
            raise ValueError('COLOR_BY = "spread" needs VARIANT = "9 samples"')
        tok_spread = np.full(len(v["P"]), np.nan); tok_spread[v["has_pos"]] = v["spread"]
        c, title = np.minimum(tok_spread[v["tok"][keep]] * 100, 50), "token spread [cm] (clipped at 50)"
    else:
        raise ValueError(COLOR_BY)
    style["marker"].update(color=c, colorscale="Viridis", colorbar=dict(title=title, thickness=12), showscale=True)
    return [go.Scatter3d(x=p[:, 0], y=p[:, 1], z=p[:, 2], name=f"{len(p):,} points", **style)]

def scene_layout(title):
    return dict(title=title, height=750, margin=dict(l=0, r=0, t=40, b=0), legend=dict(itemsizing="constant"),
                scene=dict(aspectmode="data", xaxis_visible=False, yaxis_visible=False, zaxis_visible=False,
                           camera=dict(eye=dict(x=-1.0, y=-1.3, z=1.1))))

v = variants[VARIANT]
fig = go.Figure([mesh_trace(mesh), *point_traces(v, COLOR_BY)])
fig.update_layout(**scene_layout(f"{scene.scene_id}: token positions ({VARIANT}, colored by {COLOR_BY}) on our TSDF mesh"))
fig.show()
if SAVE_HTML:
    out_dir.mkdir(parents=True, exist_ok=True)
    fig.write_html(out_dir / f"features_mesh_{VARIANT.replace(' ', '')}_{COLOR_BY}.html", include_plotlyjs="cdn")

if SHOW_O3D:
    m = o3d.geometry.TriangleMesh(mesh); m.paint_uniform_color([0.75, 0.75, 0.75]); m.compute_vertex_normals()
    keep = crop_mask(v["pts"])
    pc = o3d.geometry.PointCloud(o3d.utility.Vector3dVector(v["pts"][keep]))
    pc.colors = o3d.utility.Vector3dVector([[int(LABEL_HEX[s][i:i + 2], 16) / 255 for i in (1, 3, 5)] for s in v["label"][keep]])
    o3d.visualization.draw_geometries([m, pc], up=[0, 0, 1], front=[-0.5, -0.6, 0.6], lookat=v["pts"][keep].mean(0), zoom=0.6)

## 4. Voxel view: which voxels would become tokens?

Merge the points into `TOKEN_VOXEL` voxels the way LLaVA-3D / Qwen-3D do: every voxel that receives at least one
point becomes a token (their "occupied" voxels), with the average of its features. `COLOR_BY`:

* `"our label"` – the label **our** TSDF gives that coarse voxel (pooled as in notebook 04: occupied if any fine voxel
  is occupied/behind, unknown if > 50 % unknown, else free). Blue or gray cubes are tokens in places our map does
  not consider surface.
* `"tokens"` – how many distinct image tokens were averaged into the voxel (log scale)

`SHOW_OUR_FREE = True` adds our free voxels as small blue dots – the space that gets **no** token in their scheme,
which is what our free-space tokens add.

In [ ]:
VARIANT = "center"         # "center" or "9 samples"
COLOR_BY = "our label"     # "our label" or "tokens"
SHOW_OUR_FREE = False

pooled = pool_states(state, FACTOR)                  # coarse labels on the TOKEN_VOXEL grid (same origin)
POOLED_HEX = {OCCUPIED: LABEL_HEX[OCCUPIED], FREE: LABEL_HEX[FREE], UNKNOWN: "#9e9e9e", OUTSIDE: LABEL_HEX[OUTSIDE]}

def token_voxels(v):
    # Unique TOKEN_VOXEL indices hit by the points, the number of distinct tokens per voxel, and per-point voxel ids.
    ijk = np.floor((v["pts"] - grid.origin) / TOKEN_VOXEL).astype(int)
    pairs = np.unique(np.column_stack([ijk, v["tok"]]), axis=0)          # each token counted once per voxel
    vox, n_tok = np.unique(pairs[:, :3], axis=0, return_counts=True)
    return vox, n_tok, ijk

def coarse_label(vox):
    inside = ((vox >= 0) & (vox < np.array(pooled.shape))).all(1)
    lab = np.full(len(vox), OUTSIDE, np.uint8)
    lab[inside] = pooled[tuple(vox[inside].T)]
    return lab

_SHADE = {(2, 1): 1.0, (2, 0): 0.55, (0, 0): 0.78, (0, 1): 0.88, (1, 0): 0.7, (1, 1): 0.82}  # (axis, side) -> brightness

def cubes_trace(vox, rgb, name, size=TOKEN_VOXEL, origin=None, opacity=1.0):
    # One Mesh3d with a cube per voxel index; faces between two drawn voxels are skipped; flat shading baked in.
    origin = grid.origin if origin is None else origin
    base = vox.min(0) - 1
    dims = vox.max(0) - base + 2
    present = np.zeros(dims, bool); present[tuple((vox - base).T)] = True
    V, F, C = [], [], []
    for axis in range(3):
        a, b = [x for x in range(3) if x != axis]
        for side in (0, 1):
            nb = vox.copy(); nb[:, axis] += 1 if side else -1
            show = ~present[tuple((nb - base).T)]
            if not show.any():
                continue
            corner = vox[show].astype(float); corner[:, axis] += side
            quad = np.zeros((4, 3)); quad[[1, 2], a] = 1; quad[[2, 3], b] = 1
            verts = (corner[:, None, :] + quad[None]) * size + origin                 # (n, 4, 3)
            n0 = sum(len(x) for x in V)
            idx = n0 + 4 * np.arange(len(verts))[:, None]
            F.append(np.concatenate([idx + [0, 1, 2], idx + [0, 2, 3]]))
            V.append(verts.reshape(-1, 3))
            shade = _SHADE[(axis, side)]
            col = np.clip(rgb[show] * shade * 255, 0, 255).astype(int)
            C.append(np.concatenate([col, col]))
    V, F, C = np.concatenate(V), np.concatenate(F), np.concatenate(C)
    return go.Mesh3d(x=V[:, 0], y=V[:, 1], z=V[:, 2], i=F[:, 0], j=F[:, 1], k=F[:, 2], name=name, showlegend=True,
                     facecolor=[f"rgb({r},{g},{b})" for r, g, b in C], flatshading=True, opacity=opacity, hoverinfo="skip",
                     lighting=dict(ambient=1.0, diffuse=0.0, specular=0.0, fresnel=0.0, roughness=1.0))

def hex_rgb(h):
    return np.array([int(h[i:i + 2], 16) for i in (1, 3, 5)]) / 255

def voxel_center_z(vox):
    return grid.origin[2] + (vox[:, 2] + 0.5) * TOKEN_VOXEL

v = variants[VARIANT]
vox, n_tok, _ = token_voxels(v)
keep = np.ones(len(vox), bool) if MAX_HEIGHT is None else voxel_center_z(vox) < MAX_HEIGHT
vox, n_tok = vox[keep], n_tok[keep]
lab = coarse_label(vox)
traces = []
if COLOR_BY == "our label":
    for s in (OCCUPIED, FREE, UNKNOWN, OUTSIDE):
        m = lab == s
        if m.any():
            traces.append(cubes_trace(vox[m], np.tile(hex_rgb(POOLED_HEX[s]), (m.sum(), 1)),
                                      f"token voxel, our label: {LABEL_NAMES[s]} ({m.sum():,})"))
elif COLOR_BY == "tokens":
    t = np.log(n_tok) / np.log(n_tok.max())
    traces.append(cubes_trace(vox, plt.cm.viridis(t)[:, :3], f"token voxels ({len(vox):,}), color = log #tokens (max {n_tok.max()})"))
else:
    raise ValueError(COLOR_BY)
if SHOW_OUR_FREE:
    fc = grid.origin + (np.argwhere(pooled == FREE) + 0.5) * TOKEN_VOXEL
    fc = fc[crop_mask(fc)]
    traces.append(go.Scatter3d(x=fc[:, 0], y=fc[:, 1], z=fc[:, 2], mode="markers", name=f"our free voxels ({len(fc):,})",
                               marker=dict(size=1.5, color=LABEL_HEX[FREE], opacity=0.4)))
fig = go.Figure(traces)
fig.update_layout(**scene_layout(f"{scene.scene_id}: {TOKEN_VOXEL * 100:.0f} cm token voxels ({VARIANT}), colored by {COLOR_BY}"))
fig.show()
counts = {LABEL_NAMES[s]: int((lab == s).sum()) for s in (OCCUPIED, FREE, UNKNOWN, OUTSIDE)}
crop_note = "" if MAX_HEIGHT is None else f" below z = {MAX_HEIGHT} m"
print(f"{len(vox):,} token voxels{crop_note}; by our label: {counts}")
if SAVE_HTML:
    out_dir.mkdir(parents=True, exist_ok=True)
    fig.write_html(out_dir / f"features_voxels_{VARIANT.replace(' ', '')}_{COLOR_BY.replace(' ', '')}.html", include_plotlyjs="cdn")

## 5. Center vs. 9 samples

1. Numbers side by side.
2. **Spread** of the 9 samples per token (largest distance between two of its samples). Two effects:
   * **footprint:** a token covers ≈ 6 × 6 depth pixels, i.e. ≈ 5–10 cm of surface at 1–3 m – so even on a flat
     wall the samples are several cm apart, about one `TOKEN_VOXEL`;
   * **depth edges:** across an object boundary the samples can be meters apart, so one feature is averaged into
     voxels on **both** objects (the long tail of the histogram).
3. One frame with every token colored by its spread – edges light up.
4. Token voxels of both variants in one 3D view. With 3 × 3 samples the middle sample *is* the token center, so every
   center voxel is also a 9-sample voxel ("only center" is empty); the pink voxels are the extra ones the 9-sample
   variant creates.

In [ ]:
c, s9 = variants["center"], variants["9 samples"]
stats = {}
for name, v in variants.items():
    vox, n_tok, ijk = token_voxels(v)
    # number of distinct token voxels each token is spread over
    pairs = np.unique(np.column_stack([v["tok"], ijk]), axis=0)
    per_token = np.bincount(pairs[:, 0], minlength=len(v["P"]))[v["has_pos"]]
    lab = coarse_label(vox)
    stats[name] = {
        "tokens with a position": f"{v['has_pos'].mean() * 100:.1f} %",
        "3D points": f"{len(v['pts']):,}",
        "points in occupied ±1 voxel": f"{v['near_occ'].mean() * 100:.1f} %",
        "points in occupied ±2 voxels": f"{v['near_occ2'].mean() * 100:.1f} %",
        "points in free voxels": f"{(v['label'] == FREE).mean() * 100:.1f} %",
        "points in unknown voxels": f"{(v['label'] == UNKNOWN).mean() * 100:.1f} %",
        f"{TOKEN_VOXEL * 100:.0f} cm token voxels": f"{len(vox):,}",
        "  of which our label free / unknown": f"{(lab == FREE).sum():,} / {(lab == UNKNOWN).sum():,}",
        "token voxels per image token (mean)": f"{per_token.mean():.2f}",
        "image tokens in > 1 token voxel": f"{(per_token > 1).mean() * 100:.1f} %",
        "image tokens per token voxel (mean)": f"{n_tok.mean():.1f}",
    }
    stats[name]["_vox"] = vox
print(f"{'':40s}{'center':>14s}{'9 samples':>14s}")
for key in stats["center"]:
    if not key.startswith("_"):
        print(f"{key:40s}{stats['center'][key]:>14s}{stats['9 samples'][key]:>14s}")

sp = s9["spread"] * 100
print(f"\n9-sample spread: median {np.median(sp):.1f} cm, {np.mean(sp > 10) * 100:.1f} % of tokens > 10 cm, "
      f"{np.mean(sp > 50) * 100:.1f} % > 50 cm")

In [ ]:
FRAME = 6   # which feature frame to show (0 .. N_FEATURE_FRAMES - 1)

f = feat_frames[FRAME]
_, rgb = next(scene.iter_rgb([f.index]))
n = SAMPLES_PER_SIDE
_, (gh, gw) = token_samples((scene.rgb_height, scene.rgb_width), f.depth.shape, n)
first = FRAME * gh * gw
tok_spread = np.full(len(s9["P"]), np.nan); tok_spread[s9["has_pos"]] = s9["spread"]
img_spread = tok_spread[first:first + gh * gw].reshape(gh, gw) * 100

fig, ax = plt.subplots(1, 4, figsize=(22, 4.6), gridspec_kw=dict(width_ratios=[1.1, 1, 1, 1]))
ax[0].hist(sp[sp > 0], bins=np.logspace(-1, np.log10(max(sp.max(), 1)), 60), color="#4c72b0")
ax[0].set_xscale("log"); ax[0].set_xlabel("spread of the 9 samples [cm]"); ax[0].set_ylabel("tokens")
ax[0].axvline(TOKEN_VOXEL * 100, color="k", ls="--", lw=1, label=f"token voxel {TOKEN_VOXEL * 100:.0f} cm"); ax[0].legend()
ax[0].set_title("spread per token (all feature frames)")
ax[1].imshow(rgb); ax[1].set_title(f"frame {f.index} (RGB)")
ax[2].imshow(f.depth, cmap="turbo", vmin=0, vmax=dc["max_depth"]); ax[2].set_title("depth (invalid = 0)")
ax[3].imshow(rgb, alpha=0.45)
im = ax[3].imshow(np.minimum(img_spread, 50), cmap="magma", extent=(0, rgb.shape[1], rgb.shape[0], 0), alpha=0.75, interpolation="nearest")
plt.colorbar(im, ax=ax[3], label="spread [cm] (clipped at 50)", fraction=0.035)
ax[3].set_title(f"{gw} × {gh} tokens colored by spread")
for a in ax[1:]:
    a.axis("off")
plt.tight_layout()

In [ ]:
# Token voxels of both variants in one view.
def keyset(vox):
    return {tuple(x) for x in vox.tolist()}

kc, k9 = keyset(stats["center"]["_vox"]), keyset(stats["9 samples"]["_vox"])
groups = {"both": (kc & k9, "#7f7f7f"), "only 9 samples": (k9 - kc, "#e377c2"), "only center": (kc - k9, "#17becf")}
traces = []
for name, (keys, color) in groups.items():
    if not keys:
        continue
    vox = np.array(sorted(keys))
    if MAX_HEIGHT is not None:
        vox = vox[voxel_center_z(vox) < MAX_HEIGHT]
    traces.append(cubes_trace(vox, np.tile(hex_rgb(color), (len(vox), 1)), f"{name} ({len(vox):,})",
                              opacity=1.0 if name != "both" else 0.35))
    print(f"{name:15s} {len(keys):>7,} token voxels")
fig = go.Figure(traces)
fig.update_layout(**scene_layout(f"{scene.scene_id}: {TOKEN_VOXEL * 100:.0f} cm token voxels, center vs. 9 samples"))
fig.show()
if SAVE_HTML:
    out_dir.mkdir(parents=True, exist_ok=True)
    fig.write_html(out_dir / "features_voxels_center_vs_9.html", include_plotlyjs="cdn")

## Findings

First run, 1ada7a0617, defaults (16 feature frames, 41 × 30 tokens, 8 cm token voxels, TSDF from 276 frames):

- ~99.5 % of tokens get a position; feature points land in the same label mix as raw depth (≈ 49 % occupied,
  37 % free, 13 % behind; 82 % / 93 % within ±1 / ±2 voxels of an occupied voxel) – features sit on our surface up to fusion noise.
- 9 samples vs. center: same label mix, but **1.7× more token voxels** (15.5k vs. 9.2k), 82 % of image tokens are
  spread over > 1 token voxel (mean 2.9). Median spread 6.8 cm is mostly the token footprint; 28 % of tokens > 10 cm,
  3 % > 50 cm (depth edges).
- So with 9 samples every voxel averages more, partly unrelated features; with the center only, a token's feature
  lands in one voxel but large parts of the patch are not represented geometrically.
- Open: which `MAX_PIXELS` / frame count Qwen-3D actually uses; Peter's exact sampling (replace section 2's
  "9 samples" with his code once pushed); next step: PCA colors of the real Qwen2.5-VL features per voxel.